# M12 Stage 2：对症矩阵叠加（2000 步）

**自举**：Stage 1 最优快照（`m12s1_best.pt`）——链式第二级（v0.2 模式）
**配置**：同 Stage 1（bf16-Z + PagedAdam8bit + 检查点，CHUNK=4）
**矩阵**：wiki 2（锚，降权）+ **sciq 3（2.4ep 剂量峰值，M11 定标）** + ARC 1 + ultra 2
**缓存**：wiki 复用 Stage 1 的 Drive 缓存（前 4000 窗索引对齐）；sciq/ARC/ultra 新缓存 ~15min
**探针**：step 1000/2000 各一次（sciq 去污染 + ARC，PyTorch 似然协议）
**门**：**sciq ≥ 0.70**（M12 终局判据知识维）；ppl 相对 Stage 1 报差（分布代价）
**预算**：~4.8h ≈ 24 单元。FP 参照（已知，M10 实测 PyTorch 协议）：sciq 0.872/0.830，ARC 0.474/0.472

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
%pip -q install --force-reinstall --no-deps "transformers==4.57.1" "tokenizers==0.22.2" "huggingface-hub==0.36.2"
%pip -q install "datasets==5.0.1" "accelerate==1.14.0" sentencepiece protobuf bitsandbytes
from google.colab import drive
drive.mount("/content/drive")
import torch, transformers
print(torch.__version__, transformers.__version__)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

BLOCK = 1024
_sign_cache = {}
def signs_for_width(width, device, dtype=torch.float32):
    if width not in _sign_cache:
        g = torch.Generator().manual_seed(20260922 + width)
        _sign_cache[width] = (torch.randint(0, 2, (width,), generator=g) * 2 - 1)
    return _sign_cache[width].to(device=device, dtype=dtype)

def fwht(x):
    n = x.shape[-1]
    y = x
    h = 1
    while h < n:
        y = y.reshape(*y.shape[:-1], n // (2 * h), 2, h)
        a, b = y[..., 0, :], y[..., 1, :]
        y = torch.stack([a + b, a - b], dim=-2).reshape(*x.shape[:-1], n)
        h *= 2
    return y

def rot_fwd(x, signs, n=BLOCK):
    lead = x.shape[:-1]; w = x.shape[-1]
    xb = x.float().reshape(*lead, w // n, n)
    zb = fwht(xb * signs.reshape(-1, n)) / math.sqrt(n)
    return zb.reshape(*lead, w)

def rot_inv(x, signs, n=BLOCK):
    lead = x.shape[:-1]; w = x.shape[-1]
    xb = x.float().reshape(*lead, w // n, n)
    eb = fwht(xb) / math.sqrt(n) * signs.reshape(-1, n)
    return eb.reshape(*lead, w)

GROUP = 128
def quantize_init(w):
    out, width = w.shape
    wg = w.reshape(out, width // GROUP, GROUP)
    s = (0.836 * wg.std(dim=-1)).clamp(min=1e-8)
    t = torch.round(wg / s.unsqueeze(-1)).clamp(-1, 1).to(torch.int8)
    return t.reshape(out, width), s
print("math OK")

In [ ]:
# ===== STE（M12 Stage0 版：Z_DTYPE 可参数化、codes0 可选、backward 适配 Z dtype） =====
Z_DTYPE = torch.float32          # 臂配置覆盖为 bfloat16
KEEP_CODES0 = False              # 训练态不需要（GGUF 自举才用）

class _TernarySTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, Z, s):
        ctx.save_for_backward(Z, s)
        sr = s.repeat_interleave(GROUP, dim=1)
        return (sr * torch.clamp(torch.round(Z.float() / sr), -1, 1)).to(Z.dtype if Z.dtype==torch.bfloat16 else torch.bfloat16)

    @staticmethod
    def backward(ctx, g):
        Z, s = ctx.saved_tensors
        gf = g.float()
        sr = s.repeat_interleave(GROUP, dim=1)
        T = torch.clamp(torch.round(Z.float() / sr), -1, 1)
        mask = ((Z.float() / sr).abs() <= 1.5).to(gf.dtype)
        return (gf * sr * mask).to(Z.dtype), (gf * T).reshape(T.shape[0], -1, GROUP).sum(-1)

class _EmbSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, codes, s):
        ctx.save_for_backward(codes)
        out, width = codes.shape
        w = torch.empty(out, width, dtype=torch.bfloat16, device=codes.device)
        for a in range(0, out, 16384):
            b = min(a + 16384, out)
            w[a:b] = (s[a:b].repeat_interleave(GROUP, dim=1) * codes[a:b].float()).to(torch.bfloat16)
        return w

    @staticmethod
    def backward(ctx, g):
        (codes,) = ctx.saved_tensors
        out, width = codes.shape
        gs = torch.empty(out, width // GROUP, dtype=torch.float32, device=codes.device)
        for a in range(0, out, 16384):
            b = min(a + 16384, out)
            gs[a:b] = (g[a:b].float() * codes[a:b].float()).reshape(b - a, -1, GROUP).sum(-1)
        return None, gs

def _inv_softplus(s):
    return torch.log(torch.expm1(s.clamp(min=1e-6))).clamp(min=-12)

class RotQATLinear(nn.Module):
    def __init__(self, lin, signs):
        super().__init__()
        with torch.no_grad():
            wf = rot_fwd(lin.weight.data.float(), signs)
            codes, s = quantize_init(wf)
        self.register_buffer("signs", signs.float())
        self.Z = nn.Parameter((codes.float() * s.repeat_interleave(GROUP, 1)).to(Z_DTYPE))
        self.theta = nn.Parameter(_inv_softplus(s))
        if KEEP_CODES0:
            self.register_buffer("codes0", codes)
        self.bias = lin.bias

    def scales(self):
        return F.softplus(self.theta).clamp(min=1e-8)

    def w_eff(self):
        return _TernarySTE.apply(self.Z, self.scales())

    def forward(self, x):
        return F.linear(rot_fwd(x, self.signs).to(torch.bfloat16), self.w_eff(), self.bias)

class RotQATHead(nn.Module):
    def __init__(self, emb):
        super().__init__()
        self.emb = emb
    def forward(self, h):
        return F.linear(rot_fwd(h, self.emb.signs).to(torch.bfloat16), self.emb.w_eff())

class RotQATEmbedding(nn.Module):
    def __init__(self, table, signs):
        super().__init__()
        with torch.no_grad():
            ef = rot_fwd(table.float(), signs)
            codes, s = quantize_init(ef)
        self.register_buffer("signs", signs.float())
        self.register_buffer("codes", codes)
        self.theta = nn.Parameter(_inv_softplus(s))
    def scales(self):
        return F.softplus(self.theta).clamp(min=1e-8)
    def w_eff(self):
        return _EmbSTE.apply(self.codes, self.scales())
    def forward(self, ids):
        return rot_inv(F.embedding(ids, self.w_eff()), self.signs).to(torch.bfloat16)

class Fp32RMSNorm(nn.Module):
    def __init__(self, norm):
        super().__init__()
        self.weight = nn.Parameter(norm.weight.data.float())
        self.eps = getattr(norm, "variance_epsilon", None) or 1e-6
    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (self.weight * xf).to(x.dtype)

TARGETS = [("self_attn", n) for n in ("q_proj", "k_proj", "v_proj", "o_proj")] + \
          [("mlp", n) for n in ("gate_proj", "up_proj", "down_proj")]

def install(model):
    dev = next(model.parameters()).device
    emb = model.model.embed_tokens
    new_emb = RotQATEmbedding(emb.weight.data, signs_for_width(emb.weight.shape[1], dev))
    model.model.embed_tokens = new_emb
    model.lm_head = RotQATHead(new_emb)
    linears = []
    for li, layer in enumerate(model.model.layers):
        for ppath, name in TARGETS:
            parent = layer
            for p in ppath.split("."):
                parent = getattr(parent, p)
            lin = getattr(parent, name)
            mod = RotQATLinear(lin, signs_for_width(lin.weight.shape[1], dev))
            setattr(parent, name, mod)
            linears.append(mod)
        layer.input_layernorm = Fp32RMSNorm(layer.input_layernorm)
        layer.post_attention_layernorm = Fp32RMSNorm(layer.post_attention_layernorm)
        layer.self_attn.q_norm = Fp32RMSNorm(layer.self_attn.q_norm)
        layer.self_attn.k_norm = Fp32RMSNorm(layer.self_attn.k_norm)
        del lin
    model.model.norm = Fp32RMSNorm(model.model.norm)
    for p in model.parameters():
        p.requires_grad_(False)
    tr = {"Z": [], "theta": [], "island": []}
    for m in model.modules():
        if isinstance(m, RotQATLinear):
            m.Z.requires_grad_(True); m.theta.requires_grad_(True)
            tr["Z"].append(m.Z); tr["theta"].append(m.theta)
        elif isinstance(m, RotQATEmbedding):
            m.theta.requires_grad_(True); tr["theta"].append(m.theta)
        elif isinstance(m, Fp32RMSNorm):
            m.weight.requires_grad_(True); tr["island"].append(m.weight)
    torch.cuda.empty_cache()
    return new_emb, linears, tr
print("STE OK")

In [ ]:
# ===== 配置 =====
MODEL = "Qwen/Qwen3-8B"
STEPS, BATCH, SEQ, TOPK, CHUNK = 2000, 8, 512, 50, 4
LRS = [2e-4, 1e-3, 3e-4]
EVAL_EVERY, EVAL_WINDOWS = 100, 40
PROBE_AT = (1000, 2000)
DRIVE_DIR = "/content/drive/MyDrive/m12s1"          # 复用：s1 缓存 + 自举快照
DRIVE_DIR2 = "/content/drive/MyDrive/m12s2"
os.makedirs(DRIVE_DIR2, exist_ok=True)
CKPT_LOCAL = "/content/m12s2_last.pt"
CKPT_DRIVE_BEST = f"{DRIVE_DIR2}/m12s2_best.pt"
BOOT = f"{DRIVE_DIR}/m12s1_best.pt"                   # Stage 1 自举源
CACHE_TV_WIKI = f"{DRIVE_DIR}/cache_tv.pt"            # s1 的 40k wiki 缓存（索引对齐复用）
CACHE_TI_WIKI = f"{DRIVE_DIR}/cache_ti.pt"
CACHE_TV_MIX = f"{DRIVE_DIR2}/cache_tv_mix.pt"        # sciq/arc/ultra 三流新缓存
CACHE_TI_MIX = f"{DRIVE_DIR2}/cache_ti_mix.pt"
import torch
Z_DTYPE = torch.bfloat16
KEEP_CODES0 = False
print("config OK")

In [ ]:
# ===== 数据：wiki（确定性重建，索引对齐 s1 缓存）+ sciq/arc/ultra 窗 + 去污染 =====
import torch, gc, os
from datasets import load_dataset
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)

def ngrams(text, n=13):
    ws = text.lower().split()
    return {" ".join(ws[i:i+n]) for i in range(len(ws)-n+1)} if len(ws)>=n else set()

# wiki 窗（与 s1 完全同序——索引对齐复用缓存）
text103 = "\n\n".join(t for t in load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")["train"]["text"] if t.strip())
ids103 = tok(text103, return_tensors="pt").input_ids[0]
del text103; gc.collect()
wiki_all = [ids103[i*SEQ:(i+1)*SEQ] for i in range(min(40000, ids103.numel()//SEQ))]
del ids103; gc.collect()
wiki_src = wiki_all[:4000]          # 训练用前 4000（缓存索引 0..3999 已覆盖）
text2 = "\n\n".join(t for t in load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")["test"]["text"] if t.strip())
test_ids = tok(text2, return_tensors="pt").input_ids[0]

# sciq（对症主探针流）+ 去污染评估集
sciq_tr = load_dataset("allenai/sciq", split="train")
s_ids = []
for ex in sciq_tr:
    s_ids.append(tok(f"Question: {ex['question']}\n{ex['support']}\nAnswer: {ex['correct_answer']}",
                     return_tensors="pt").input_ids[0])
_all_sciq = torch.cat(s_ids); del s_ids; gc.collect()
q_all = _all_sciq[: (_all_sciq.numel() // SEQ) * SEQ]; del _all_sciq; gc.collect()
sciq_src = [q_all[i*SEQ:(i+1)*SEQ] for i in range(q_all.numel()//SEQ)]

_train_ng = set()
for ex in sciq_tr:
    _train_ng |= ngrams(ex["support"] + " " + ex["question"])
sciq_val_all = load_dataset("allenai/sciq", split="validation")
CLEAN_IDX = [i for i, ex in enumerate(sciq_val_all)
             if not (ngrams(ex["support"] + " " + ex["question"]) & _train_ng)]
sciq_val = sciq_val_all.select(CLEAN_IDX)
del _train_ng; gc.collect()

# ARC（对症第二流）
arc_c = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="train")
arc_e = load_dataset("allenai/ai2_arc", "ARC-Easy", split="train")
a_ids = []
for ex in list(arc_c) + list(arc_e):
    try:
        ai = ex["choices"]["label"].index(ex["answerKey"])
        ans = ex["choices"]["text"][ai]
    except ValueError:
        continue
    a_ids.append(tok(f"Question: {ex['question']}\nAnswer: {ans}", return_tensors="pt").input_ids[0])
_all_arc = torch.cat(a_ids); del a_ids; gc.collect()
a_all = _all_arc[: (_all_arc.numel() // SEQ) * SEQ]; del _all_arc; gc.collect()
arc_src = [a_all[i*SEQ:(i+1)*SEQ] for i in range(a_all.numel()//SEQ)]
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

# ultra（生成保持，2/step）
import numpy as np
rng = np.random.default_rng(0)
uc = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").select(range(8000))
u_ids = []
for ex in uc:
    text = tok.apply_chat_template(ex["messages"], tokenize=False,
                                   chat_template_kwargs={"enable_thinking": False})
    u_ids.append(tok(text, return_tensors="pt").input_ids[0])
u_all = torch.cat([i[: (i.numel() // SEQ) * SEQ] for i in u_ids if i.numel() >= SEQ])
del u_ids; gc.collect()
_u_off = rng.choice(u_all.numel() // SEQ, min(4000, u_all.numel() // SEQ), replace=False)
ultra_src = [u_all[int(o)*SEQ: int(o)*SEQ+SEQ] for o in _u_off]
del u_all; gc.collect()

os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["HF_DATASETS_OFFLINE"] = "1"
print(f"wiki {len(wiki_src)} | sciq {len(sciq_src)} | arc {len(arc_src)} | ultra {len(ultra_src)} | "
      f"sciq_val {len(sciq_val)} | ARC test {len(arc)}")

In [ ]:
# ===== 缓存：wiki 复用 + 三流新缓存 → FP 参照 ppl → install → 自举 s1 快照 =====
import time, gc, math
import torch.nn.functional as F
from transformers import AutoModelForCausalLM

DEV = "cuda"
tv_wiki = torch.load(CACHE_TV_WIKI); ti_wiki = torch.load(CACHE_TI_WIKI)
print(f"wiki 缓存复用: {len(tv_wiki)} 窗（训练用前 4000）", flush=True)

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16,
                                             attn_implementation="sdpa").to(DEV)

# ---- 探针定义（PyTorch 似然协议，与全部 1.7B 实验一致）----
Q, CH = "Question: {q}\nAnswer:", " {a}"
@torch.no_grad()
def _score(prompt, option):
    ids = tok(prompt + CH.format(a=option), return_tensors="pt").input_ids[0]
    pl = tok(prompt, return_tensors="pt").input_ids[0].numel()
    x = ids.unsqueeze(0).to(DEV)
    lg = model(x).logits.float()[0, pl-1:-1]
    tgt = ids[pl:].to(DEV)
    return (torch.log_softmax(lg, -1)[torch.arange(len(tgt), device=DEV), tgt].sum().item(), len(tgt))

@torch.no_grad()
def probe(data, tag, get_opts, get_gold, m=None):
    mm = m or model
    mm.eval(); n = acc = accn = 0
    for ex in data:
        prompt = Q.format(q=ex["question"]) + "\n"
        sc = [_score(prompt, o) for o in get_opts(ex)]
        acc += max(range(len(sc)), key=lambda i: sc[i][0]) == get_gold(ex)
        accn += max(range(len(sc)), key=lambda i: sc[i][0]/sc[i][1]) == get_gold(ex)
        n += 1
    mm.train()
    print(f"[{tag}] acc {acc/n:.3f} | acc_norm {accn/n:.3f} (n={n})", flush=True)

# ---- 三流新缓存（sciq/arc/ultra 拼接索引：0..S-1 sciq, S..S+A-1 arc, 之后 ultra）----
S, A = len(sciq_src), len(arc_src)
mix_src = sciq_src + arc_src + ultra_src
need = not (os.path.exists(CACHE_TV_MIX) and os.path.exists(CACHE_TI_MIX))
if need:
    t0 = time.time()
    tvm = torch.empty(len(mix_src), SEQ - 1, TOPK, dtype=torch.float16, pin_memory=True)
    tim = torch.empty(len(mix_src), SEQ - 1, TOPK, dtype=torch.int32, pin_memory=True)
    with torch.no_grad():
        for c0 in range(0, len(mix_src), 8):
            x = torch.stack(mix_src[c0:c0+8]).to(DEV)
            lg = model(x).logits[:, :-1].float()
            v, i = torch.topk(lg, TOPK, dim=-1)
            tvm[c0:c0+len(v)] = v.half().cpu(); tim[c0:c0+len(i)] = i.cpu()
            del lg, v, i
            if (c0 // 8) % 100 == 0:
                print(f"  mix cache {c0+8}/{len(mix_src)} ({(time.time()-t0)/60:.0f}min)", flush=True)
    torch.save(tvm, CACHE_TV_MIX); torch.save(tim, CACHE_TI_MIX)
    print(f"mix cache done {(time.time()-t0)/60:.0f}min -> Drive", flush=True)
else:
    tvm = torch.load(CACHE_TV_MIX); tim = torch.load(CACHE_TI_MIX)
    print(f"mix cache reloaded: {len(tvm)}", flush=True)

@torch.no_grad()
def ppl_of(m, n_windows=EVAL_WINDOWS, seq=SEQ):
    m.eval(); nll = cnt = skipped = 0
    V = m.config.vocab_size
    for w in range(n_windows):
        x = test_ids[w*seq:(w+1)*seq].unsqueeze(0).to(DEV)
        lg = m(x).logits.float()
        v = F.cross_entropy(lg[:, :-1].reshape(-1, V), x[:, 1:].reshape(-1), reduction="sum")
        if torch.isfinite(v): nll += v.item(); cnt += x[:, 1:].numel()
        else: skipped += 1
    m.train()
    return (math.exp(nll/cnt) if cnt else float("nan")), skipped

fp_ppl, _ = ppl_of(model)
print(f"FP-8B reference ppl = {fp_ppl:.2f}（M12 终局门 = {fp_ppl*1.2:.2f}）", flush=True)

# ---- install + 检查点 + 优化器（Stage 0 定案配方）----
emb, linears, tr = install(model)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
import bitsandbytes as bnb
opt = bnb.optim.PagedAdam8bit([{"params": tr["Z"], "lr": LRS[0]},
                               {"params": tr["theta"], "lr": LRS[1]},
                               {"params": tr["island"], "lr": LRS[2]}], betas=(0.9, 0.95))
gc.collect(); torch.cuda.empty_cache()
print(f"installed {len(linears)} linears | ckpt ON | VRAM {torch.cuda.memory_allocated()/2**30:.1f}GB", flush=True)

# ---- 自举：Stage 1 最优快照（冷优化器，链式第二级）----
import os as _os
_boot_from = BOOT
if _os.path.exists(CKPT_DRIVE_BEST):
    ck_a = torch.load(CKPT_DRIVE_BEST, map_location="cpu")
    ck_b = torch.load(BOOT, map_location="cpu")
    _boot_from = CKPT_DRIVE_BEST if ck_a.get("step", 0) >= getattr(ck_b, "step", ck_b.get("step", 0)) else BOOT
_start, hist = 1, []
if _os.path.exists(_boot_from):
    ck = torch.load(_boot_from, map_location="cpu")
    with torch.no_grad():
        for m_, Z_, th_ in zip(linears, ck["Z"], ck["theta"]):
            m_.Z.copy_(Z_.to(DEV, dtype=torch.bfloat16)); m_.theta.copy_(th_)
        emb.theta.copy_(ck["emb_theta"]); emb.codes.copy_(ck["emb_codes"])
        for n_, m_i in model.named_modules():
            if isinstance(m_i, Fp32RMSNorm) and n_ in ck["islands"]:
                m_i.weight.copy_(ck["islands"][n_])
    _step0 = ck.get("step", 0)
    print(f"bootstrapped from {_boot_from} (source step {_step0}, fresh optimizer)", flush=True)
if _os.path.exists(CKPT_DRIVE_BEST) and torch.load(CKPT_DRIVE_BEST, map_location="cpu").get("arm", "") == "m12s2" \
   and torch.load(CKPT_DRIVE_BEST, map_location="cpu").get("step", 0) < STEPS:
    ck = torch.load(CKPT_DRIVE_BEST, map_location="cpu")
    _start = ck["step"] + 1; hist = ck.get("hist", [])
    print(f"RESUME m12s2 from step {ck['step']}", flush=True)
boot_ppl, _ = ppl_of(model)
print(f"bootstrap restore ppl = {boot_ppl:.2f}（应接近 Stage1 best）", flush=True)

In [ ]:
# ===== 训练循环（v2：每 500 步无条件 Drive 快照——断连零损失 + 剂量点留档）=====
import time, math, os
import torch.nn.functional as F
import shutil as _shutil, time as _time

def snapshot(path, step_i, ppl_i):
    # 容错快照：本地中转 -> 拷贝（重试x2）-> 失败只告警不杀训练
    def _state():
        return {"step": step_i, "ppl": ppl_i, "arm": "m12s2", "hist": hist,
                "Z": [m.Z.detach().cpu() for m in linears],
                "theta": [m.theta.detach().cpu() for m in linears],
                "emb_theta": emb.theta.detach().cpu(), "emb_codes": emb.codes.cpu(),
                "islands": {n: m.weight.detach().cpu() for n, m in model.named_modules()
                            if isinstance(m, Fp32RMSNorm)}}
    try:
        if path.startswith("/content/drive"):
            tmp = "/content/m12s2_snap_tmp.pt"
            torch.save(_state(), tmp)
            for attempt in range(2):
                try:
                    _shutil.copy(tmp, path); break
                except Exception as e:
                    print(f"!! Drive 拷贝重试 {attempt+1}: {type(e).__name__}", flush=True)
                    _time.sleep(20)
            else:
                print(f"!! snapshot 两次拷贝均失败——训练继续", flush=True)
                return
            os.remove(tmp)
        else:
            torch.save(_state(), path)
    except Exception as e:
        print(f"!! snapshot 失败（{type(e).__name__}）——训练继续", flush=True)

_start = 1
if os.path.exists(CKPT_DRIVE_BEST):
    _ck = torch.load(CKPT_DRIVE_BEST, map_location="cpu")
    if _ck.get("arm", "") == "m12s2" and _ck.get("step", 0) < STEPS:
        _start = _ck["step"] + 1
        hist = _ck.get("hist", [])
        print(f"RESUME m12s2 from step {_ck['step']}", flush=True)
    elif _ck.get("arm", "") == "m12s2":
        _start = STEPS + 1
        print(f"m12s2 快照已完备（step {_ck['step']}）——训练跳过", flush=True)

model.train()
best = boot_ppl
t0 = time.time()
for step in range(_start, STEPS + 1):
    j = step - 1
    # 矩阵配比：wiki 2 | sciq 3 | arc 1 | ultra 2
    idx = ([(j*2+k) % 4000 for k in range(2)] +
           [S - 1 - ((j*3+k) % S) for k in range(3)] +
           [S + (j % A)] +
           [S + A + ((j*2+k) % len(ultra_src)) for k in range(2)])
    wtok = [(tv_wiki[(j*2+k) % 4000], ti_wiki[(j*2+k) % 4000]) for k in range(2)]
    xs = [wiki_src[(j*2+k) % 4000] for k in range(2)] + [mix_src[k] for k in idx[2:]]
    tgts = wtok + [(tvm[k], tim[k]) for k in idx[2:]]
    opt.zero_grad(set_to_none=True)
    loss_val = 0.0
    for c in range(0, BATCH, CHUNK):
        n = min(CHUNK, BATCH - c)
        x = torch.stack(xs[c:c+n]).to(DEV)
        tv_c = torch.stack([tgts[c+k][0] for k in range(n)]).to(DEV).float()
        ti_c = torch.stack([tgts[c+k][1] for k in range(n)]).to(DEV).long()
        logits = model(x).logits[:, :-1].float()
        s_logp = F.log_softmax(logits, -1)
        loss_c = -(F.softmax(tv_c, -1) * torch.gather(s_logp, -1, ti_c)).sum(-1).mean()
        (loss_c * n / BATCH).backward()
        loss_val += loss_c.item() * n / BATCH
        del logits, s_logp, loss_c
    torch.nn.utils.clip_grad_norm_(
        [p for p in tr["Z"] + tr["theta"] + tr["island"] if p.grad is not None], 1.0)
    opt.step()
    if step % 20 == 0:
        print(f">> [m12s2] step {step} loss {loss_val:.3f} ({step*BATCH*SEQ/(time.time()-t0):.0f} tok/s)", flush=True)
    if step % 500 == 0:
        snapshot(CKPT_DRIVE_BEST, step, None)   # 无条件：断连零损失 + 剂量点留档
    if step % EVAL_EVERY == 0 or step == STEPS:
        ppl, skipped = ppl_of(model)
        hist.append((step, ppl))
        print(f">> [m12s2] step {step} ppl={ppl:.2f} skip={skipped}", flush=True)
    if step in PROBE_AT:
        probe(sciq_val, f"DOSE sciq @{step} (FP 0.830)",
              lambda ex: [ex["correct_answer"]] + [ex["distractor" + str(i)] for i in (1, 2, 3)],
              lambda ex: 0)
        probe(arc, f"DOSE ARC @{step} (FP 0.472)",
              lambda ex: ex["choices"]["text"],
              lambda ex: ex["choices"]["label"].index(ex["answerKey"]))
print("DONE Stage 2（500/1000/1500/2000 各有剂量点快照在 Drive）")

In [ ]:
# ===== 终局判卷 =====
probe(sciq_val, "FINAL sciq (FP 0.751/0.830)",
      lambda ex: [ex["correct_answer"]] + [ex["distractor" + str(i)] for i in (1, 2, 3)],
      lambda ex: 0)
probe(arc, "FINAL ARC (FP 0.357/0.472 ... 实测 0.474/0.472)",
      lambda ex: ex["choices"]["text"],
      lambda ex: ex["choices"]["label"].index(ex["answerKey"]))
ppl_f, _ = ppl_of(model)
print()
print("=== M12 终局判读 ===")
print(f"ppl: {ppl_f:.2f}（FP-8B {fp_ppl:.2f} 的 {ppl_f/fp_ppl:.3f}x；FP-1.7B 20.42）")
print("判据：sciq acc_norm >= 0.70 -> M12 知识维达标（'三值 8B 全面超 FP-1.7B' 的知识半壁）")
print("      ppl 对照 FP-1.7B 的 20.42：低于它则分布维也达标")
print(f"\nhistory: {hist}")

In [ ]:
# ===== 抽检（生成保持）=====
QUESTIONS = ["What planet is known as the Red Planet?",
             "Why do we see lightning before we hear thunder?",
             "What is the powerhouse of the cell?"]
@torch.no_grad()
def ask(q, max_new=96):
    ids = tok.apply_chat_template([{"role": "user", "content": q}], tokenize=True,
                                  add_generation_prompt=True,
                                  chat_template_kwargs={"enable_thinking": False},
                                  return_tensors="pt").to(DEV)
    out = model.generate(ids, max_new_tokens=max_new, do_sample=True,
                         temperature=0.5, top_p=0.85, top_k=20,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
model.eval()
for q in QUESTIONS:
    print("Q:", q, "\nA:", ask(q), "\n", flush=True)